# Shoplifting Detection: Exploratory Data Analysis & Integrity Check

This notebook performs a rigorous deep dive into the dataset integrity, duplicate structure, video characteristics (frame count, FPS, resolution, duration), and grouped stratified splitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load generated metadata
metadata_path = Path("../outputs/metadata.csv")
df = pd.read_csv(metadata_path)
print(f"Total videos indexed: {len(df)}")
df.head()

## 1. Duplicate & Integrity Analysis

Investigation into `_1` suffixes in normal video clips:
- We computed MD5 hashes across all 855 videos.
- 218 files in `non shop lifters` are exact byte-for-byte duplicates of their base clips.
- 0 duplicates exist in `shop lifters`.
- True underlying dataset size: 637 unique videos (313 normal, 324 shoplifting), almost perfectly balanced (49.1% vs 50.9%).

In [ ]:
dup_summary = df.groupby(["label_name", "is_duplicate"]).size().unstack(fill_value=0)
dup_summary["Total"] = dup_summary.sum(axis=1)
dup_summary["Unique Videos"] = dup_summary[False]
dup_summary["Duplicate Clones"] = dup_summary[True]
dup_summary[["Unique Videos", "Duplicate Clones", "Total"]]

In [ ]:
# Summary of video properties (Duration, Frame count, FPS, Resolution)
print("--- Video Properties Summary ---")
print(df.groupby("label_name")[["n_frames", "fps", "duration_s", "file_size"]].describe().T)

print("\n--- Resolutions ---")
print(df["width"].astype(str) + "x" + df["height"].astype(str).value_counts())

print("\n--- Split Breakdown (All 855 clips) ---")
print(pd.crosstab(df["split"], df["label_name"], margins=True))

print("\n--- Split Breakdown (637 Unique Events) ---")
unique_df = df[~df["is_duplicate"]]
print(pd.crosstab(unique_df["split"], unique_df["label_name"], margins=True))

In [ ]:
from IPython.display import Image, display

eda_dir = Path("../outputs/eda")
for plot_name in ["class_distribution.png", "duration_distribution.png", "splits_distribution.png", "sample_frames_grid.png"]:
    img_path = eda_dir / plot_name
    if img_path.exists():
        print(f"Displaying: {plot_name}")
        display(Image(filename=str(img_path)))